# Demostración del Orquestador Multi-Agente de Análisis e Investigación con RAG

Este notebook demuestra el funcionamiento del orquestador multi-agente en LangGraph.
El flujo incluye:
1. **Supervisor / Router Inteligente**: Coordina la delegación.
2. **Agente Investigador**: Realiza búsquedas vectoriales sobre la base documental RAG en Español.
3. **Agente Analista / Sintetizador**: Procesa los hallazgos y redacta la respuesta final en el idioma destino configurado.

In [ ]:
import os
import asyncio
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from graph import graph

load_dotenv()

## Ejemplo 1: Consulta documental en Español
Preguntamos acerca de Kanbun Uechi y la historia del Karate Uechi-Ryu.

In [ ]:
async def probar_orquestador(pregunta: str, idioma: str = "Español"):
    print(f"=== CONSULTA: {pregunta} (Idioma Destino: {idioma}) ===\n")
    initial_state = {
        "messages": [HumanMessage(content=pregunta)],
        "idioma_destino": idioma,
        "iterations": 0
    }
    async for event in graph.astream(initial_state, stream_mode="updates"):
        for node_name, state_update in event.items():
            print(f"[NODO: {node_name.upper()}]")
            if "next" in state_update:
                print(f"  -> Decisión Supervisor: {state_update['next']}")
            if "messages" in state_update:
                for msg in state_update["messages"]:
                    sender = getattr(msg, "name", "desconocido")
                    print(f"  -> [{sender}]:\n{msg.content}\n")

await probar_orquestador("¿Quién fue Kanbun Uechi y qué estilo de Karate desarrolló?", "Español")

## Ejemplo 2: Consulta con traducción/síntesis en Francés
Investigación en la base en español y síntesis adaptada al Francés.

In [ ]:
await probar_orquestador("¿Cuáles son las tres formas/katas principales traídas de China por Kanbun Uechi?", "Francés")

## Ejemplo 3: Pregunta trampa (sin información disponible)
Verificación de que el modelo no alucina información no presente en la base de datos.

In [ ]:
await probar_orquestador("¿En qué año nació Napoleón Bonaparte?", "Español")